Importaremos los archivos para compararlos 

A. Plantea la comparación
Formula una predicción o expectativa antes de calcular.
Identifica el periodo común y las variables que compararás.

Prediccion 
AUnque shangai es linea costera, y medellin esta por encima del nivel del mar, shangai presenta mas fluctuaciones de temperatura, al tener estaciones. 
Mientras que medellin, suele tener temperaturas muy similares todo el a;o.

Espero que
- Temperatura en medellin sea promedio de 22 grados 
- Variacion en el año debe ser mas en shagai al tener estaciones
- La diferencia entre maxima y mina debe ser mayor en medellin, al tener un rango mas aplio y estar cerca al ecuador

Vamos a usar el rango de años del 2019-2021 a que tenemos datos de ambas cuidades
y se usaran las columnas: temperatura maxia, temperatura minima 



Librerias 

In [2]:
import pandas as pd


In [6]:
import os
print(os.getcwd())
print(os.listdir("datos"))

c:\Users\aaram\Documents\Primer semestre Ingenierria de software\Analitica de datos\Act2\Comparaci-n_datos_clim-ticos
['medellin.csv', 'shanghai.csv']


In [ ]:
#leer ambos archivos csv
med = pd.read_csv("datos/medellin.csv")
sha = pd.read_csv("datos/shanghai.csv")

#le diremos que comlumnas debe leer en este caso, para homologar ambos archivos y poder compararlos 
med = med.rename(columns={"Date (yyyy-mm-dd)": "fecha", "Max_temperature (°C)": "t_max",
                          "Min_temperature (°C)": "t_min", "Rain (mm)": "lluvia",
                          "Wind (km/h)": "viento"})
sha = sha.rename(columns={"date_time": "fecha", "maxtempC": "t_max", "mintempC": "t_min",
                          "precipMM": "lluvia", "windspeedKmph": "viento"})


#para el analisis usaremos las siguientes columnas, le asignaremos el nombre completo de la cuidad a ambos archvios 
cols = ["fecha", "t_max", "t_min", "lluvia", "viento"]
#le asignaremos el nombre completo de la cuidad a ambos archvios 
med = med[cols].assign(ciudad="Medellín")
sha = sha[cols].assign(ciudad="Shanghái")

#desde el dataframe para medelllin(med) y shangai (sha), formateremos esta columna para que quede en tipo de datos datetime en ambos DF
for df in (med, sha):
    df["fecha"] = pd.to_datetime(df["fecha"])

#vamos a comparar los datos que tengan la msma fecha definiendo un periodo de 2019-01-01 a 2021-01-30 para nuestro DF shangai 
#Esto para tener un patron de comparacion similar al DF de medellin
inicio = max(med.fecha.min(), sha.fecha.min())   # 2019-01-01
fin    = min(med.fecha.max(), sha.fecha.max())   # 2021-01-30

#vamos a concatenar los datos, por medio de un query, filtrando datos a cuidad y fecha 
datos = (pd.concat([med, sha])
           .query("@inicio <= fecha <= @fin")
           .sort_values(["ciudad", "fecha"]))

#hagamos una validacion de print, para ver la fecha de inicio y la fecha final.
print(inicio.date(), "→", fin.date())
#agruparemos los datos por cuidad 
print(datos.groupby("ciudad").size())

2019-01-01 → 2021-01-30
ciudad
Medellín    761
Shanghái    761
dtype: int64


Revisemos la calidad de los datos encontrados 

In [ ]:
#imprimir desde la agrupacion de cuidad la temperatura maxima, la minima y el viento
print(datos.groupby("ciudad")[["t_max", "t_min", "lluvia", "viento"]].describe().T)

#
raras = datos[(datos.ciudad == "Medellín") & (datos.t_min < 12)]
print(len(raras), "días de Medellín con mínima bajo 12 °C")

ciudad          Medellín    Shanghái
t_max  count  761.000000  761.000000
       mean    22.357424   21.596583
       std      2.631485    8.556404
       min     15.000000   -3.000000
       25%     21.000000   14.000000
       50%     22.000000   22.000000
       75%     24.000000   29.000000
       max     29.000000   37.000000
t_min  count  761.000000  761.000000
       mean    11.638633   15.664915
       std      1.881917    8.063200
       min      5.000000   -6.000000
       25%     10.000000    8.000000
       50%     12.000000   17.000000
       75%     13.000000   22.000000
       max     17.000000   31.000000
lluvia count  761.000000  761.000000
       mean    20.254139    3.363075
       std     14.509451    9.822780
       min      0.000000    0.000000
       25%      9.300000    0.000000
       50%     18.200000    0.100000
       75%     28.600000    2.100000
       max     79.100000  126.800000
viento count  761.000000  761.000000
       mean     3.969777   12.329829
 

Aunque  vemos temprearutras or debajo de los 12 grados, podemos intuir que uno de los puntos analizados en estas medicciones puede corresponder a Rionegro, donde las temperaturas en la noche y madrugadas si pueden alcanzar estos rangos 